# PDF Parsing — pypdf

First half of the split pypdf pipeline. Reads PDFs from the `rag_chatbot_dev.bronze.docs` volume and extracts their text with the `pypdf` library, persisting the raw text to `rag_chatbot_dev.bronze.doc_parsed_pypdf`.

The same content-hash cache as the built-in pipeline applies: unchanged files are not re-extracted and rows of files removed from the volume are purged. (pypdf extraction is free and local — the cache here is for symmetry and consistency, not cost.) The companion notebook `PDF Chunking (pypdf)` turns this table into chunks.

In [0]:
%pip install pypdf

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# --- Configuration -------------------------------------------------------
CATALOG = "rag_chatbot_dev"
SCHEMA = "bronze"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/docs"        # source volume containing the PDF files
PARSE_TABLE = f"{CATALOG}.{SCHEMA}.doc_parsed_pypdf"     # raw text cache written by this notebook

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {PARSE_TABLE} (
    doc_path     STRING NOT NULL,
    doc_name     STRING,
    content_hash STRING,
    text         STRING,
    parsed_at    TIMESTAMP,
    CONSTRAINT doc_parsed_pypdf_pk PRIMARY KEY (doc_path)
);
""")

# Every PDF currently in the volume, with a content hash
spark.sql(f"""
    WITH files AS (
        SELECT regexp_replace(path, '^(file://|dbfs:)', '') AS doc_path, content
        FROM READ_FILES('{VOLUME_PATH}/*.pdf', format => 'binaryFile')
    )
    SELECT
        doc_path,
        element_at(split(doc_path, '/'), size(split(doc_path, '/'))) AS doc_name,
        sha2(content, 256) AS content_hash,
        content
    FROM files
""").createOrReplaceTempView("volume_files")

spark.table(PARSE_TABLE).createOrReplaceTempView("parsed_cache")

# Cache misses: files that are new, or whose content changed since the last
# extraction (LEFT ANTI JOIN keeps only rows with no exact doc_path + hash match)
spark.sql("""
    SELECT f.*
    FROM volume_files f
    LEFT ANTI JOIN parsed_cache c
        ON f.doc_path = c.doc_path AND f.content_hash = c.content_hash
""").createOrReplaceTempView("files_to_parse")

num_total = spark.table("volume_files").count()
num_to_parse = spark.table("files_to_parse").count()
print(f"{num_total} PDF(s) in volume — {num_to_parse} new or changed, {num_total - num_to_parse} served from text cache")

4 PDF(s) in volume — 0 new or changed, 4 served from text cache


In [0]:
import io

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf
from pypdf import PdfReader


@pandas_udf("string")
def extract_text(contents: pd.Series) -> pd.Series:
    """Extract the full text of one PDF (pages joined with newlines)."""
    texts = []
    for binary in contents:
        reader = PdfReader(io.BytesIO(binary))
        texts.append("\n".join(page.extract_text() or "" for page in reader.pages))
    return pd.Series(texts)


# Cache-aware refresh: only new or changed files are re-extracted; rows of
# files removed from the volume are purged.
changed_paths = [r.doc_path for r in spark.table("files_to_parse").select("doc_path").collect()]
all_paths = [r.doc_path for r in spark.table("volume_files").select("doc_path").collect()]

if changed_paths:
    # Drop superseded extractions before inserting fresh ones (PK is doc_path)
    changed_sql = ", ".join(f"'{p.replace(chr(39), chr(39) * 2)}'" for p in changed_paths)
    spark.sql(f"DELETE FROM {PARSE_TABLE} WHERE doc_path IN ({changed_sql})")

    # pandas UDFs are not visible to spark.sql — apply via the DataFrame API
    (
        spark.table("files_to_parse")
        .select(
            "doc_path",
            "doc_name",
            "content_hash",
            extract_text(F.col("content")).alias("text"),
            F.current_timestamp().alias("parsed_at"),
        )
        .write.mode("append")
        .saveAsTable(PARSE_TABLE)
    )

if all_paths:
    # Purge extractions of documents no longer in the volume
    all_sql = ", ".join(f"'{p.replace(chr(39), chr(39) * 2)}'" for p in all_paths)
    spark.sql(f"DELETE FROM {PARSE_TABLE} WHERE doc_path NOT IN ({all_sql})")
else:
    spark.sql(f"DELETE FROM {PARSE_TABLE} WHERE TRUE")

In [0]:
from pyspark.sql import functions as F

print(f"Total documents cached in {PARSE_TABLE}: {spark.table(PARSE_TABLE).count()}")

empty = spark.table(PARSE_TABLE).where(F.length("text") == 0).count()
print(f"Parses with no extractable text: {empty}")

display(spark.table(PARSE_TABLE).select("doc_name", F.length("text").alias("text_chars"), "parsed_at"))

Total documents cached in rag_chatbot_dev.bronze.doc_parsed_pypdf: 4
Parses with no extractable text: 0


doc_name,text_chars,parsed_at
expense_policy.pdf,919,2026-11-07T16:22:22.386Z
hr_leave_policy.pdf,1021,2026-11-07T16:22:22.386Z
incident_runbook.pdf,1118,2026-11-07T16:22:22.386Z
product_faq.pdf,1064,2026-11-07T16:22:22.386Z
